In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% ! important;}
div.cell.code_cell.rendered{width:100%}
div.input_prompt{padding:0px}
div.CodeMirror {font-family:Consolas ; font-size:12pt;}
div.text_cell_render.rendered_html {font-size:12pt;}
div.output {font-size:12pt; font-weight:bold}
div.input {font-family:Consolas ; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper {padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

# 벡터 DB : Chroma VS Pinecone
- Chroma : 인메모리 vector DB, 로컬 vector DB
- Pinecone : 클라우드 vector DB
    (https://www.pinecone.io/ 에서 api key 생성 -> .env에 추가(PINECONE_API_KEY 등록)

# 0. 패키지 설치

In [3]:
%pip install -q pinecone langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


# 1. knowledge Base 구성을 위한 데이터 생성

In [2]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('data/소득세법_with_markdown.docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)
document_list=loader.load_and_split(text_splitter=text_splitter)
len(document_list)

194

In [3]:
print(document_list[46].page_content)

제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>



| 종합소득 과세표준 | 세율 |

|---|---|

| 1,400만원 이하 | 과세표준의 6퍼센트 |

| 1,400만원 초과 5,000만원 이하 | 84만원 + (1,400만원을 초과하는 금액의 15퍼센트) |

| 5,000만원 초과 8,800만원 이하 | 624만원 + (5,000만원을 초과하는 금액의 24퍼센트) |

| 8,800만원 초과 1억5천만원 이하 | 1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트) |

| 1억5천만원 초과 3억원 이하 | 3,706만원 + (1억5천만원을 초과하는 금액의 38퍼센트) |

| 3억원 초과 5억원 이하 | 9,406만원 + (3억원을 초과하는 금액의 40퍼센트) |

| 5억원 초과 10억원 이하 | 1억7,406만원 + (5억원을 초과하는 금액의 42퍼센트) |

| 10억원 초과 | 3억8,406만원 + (10억원을 초과하는 금액의 45퍼센트) |





② 거주자의 퇴직소득에 대한 소득세는 다음 각 호의 순서에 따라 계산한 금액(이하 “퇴직소득 산출세액”이라 한다)으로 한다.<개정 2013. 1. 1., 2014. 12. 23.>

1. 해당 과세기간의 퇴직소득과세표준에 제1항의 세율을 적용하여 계산한 금액

2. 제1호의 금액을 12로 나눈 금액에 근속연수를 곱한 금액

3. 삭제<2014. 12. 23.>

[전문개정 2009. 12. 31.]



제2관 세액공제 <개정 2009. 12. 31.>



제56조(배당세액공제) ① 거주자의 종합소득금액에 제17조제3항 각 호 외의 부분 단서가 적용되는 배당소득금액이 합산되어 있는 경우에는 같은 항 각 호 외의 부분 단

In [4]:
# embedding : upstage의 solar-embedding-1-large-passage
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embedding = UpstageEmbeddings(
    #model="solar-embedding-1-large-passage"
    model="solar-embedding-1-large"
)

In [5]:
len(embedding.embed_query('소득세법'))

4096

In [6]:
%%time
#pinecone vector database 저장
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
import os
pc= Pinecone(
    api_key=os.getenv("PINECONE_API_KEY")
)
index_name="tax-index-markdown"

# 데이터를 처음 업로드할 때 
# database = PineconeVectorStore.from_documents(
#     documents = document_list,
#     embedding= embedding,
#     index_name=index_name
# )

# 업로드시 경고가 안보이려면 아나콘다 프롬프트 llm 환경에서 conda install -c conda-forge ipywidgets

# 업로드한 벡터db를 가져올 때
database = PineconeVectorStore(
    embedding=embedding, # 질문을 임베딩하여 유사도 검색
    index_name=index_name
)

CPU times: total: 8.05 s
Wall time: 52.3 s


# 2. 답변 생성전 retrieval 확인

In [7]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrueved_docs = database.similarity_search(query, k=3) # 기본k값:4

In [11]:
print(retrueved_docs[1].page_content)

제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>



| 종합소득 과세표준 | 세율 |

|---|---|

| 1,400만원 이하 | 과세표준의 6퍼센트 |

| 1,400만원 초과 5,000만원 이하 | 84만원 + (1,400만원을 초과하는 금액의 15퍼센트) |

| 5,000만원 초과 8,800만원 이하 | 624만원 + (5,000만원을 초과하는 금액의 24퍼센트) |

| 8,800만원 초과 1억5천만원 이하 | 1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트) |

| 1억5천만원 초과 3억원 이하 | 3,706만원 + (1억5천만원을 초과하는 금액의 38퍼센트) |

| 3억원 초과 5억원 이하 | 9,406만원 + (3억원을 초과하는 금액의 40퍼센트) |

| 5억원 초과 10억원 이하 | 1억7,406만원 + (5억원을 초과하는 금액의 42퍼센트) |

| 10억원 초과 | 3억8,406만원 + (10억원을 초과하는 금액의 45퍼센트) |





② 거주자의 퇴직소득에 대한 소득세는 다음 각 호의 순서에 따라 계산한 금액(이하 “퇴직소득 산출세액”이라 한다)으로 한다.<개정 2013. 1. 1., 2014. 12. 23.>

1. 해당 과세기간의 퇴직소득과세표준에 제1항의 세율을 적용하여 계산한 금액

2. 제1호의 금액을 12로 나눈 금액에 근속연수를 곱한 금액

3. 삭제<2014. 12. 23.>

[전문개정 2009. 12. 31.]



제2관 세액공제 <개정 2009. 12. 31.>



제56조(배당세액공제) ① 거주자의 종합소득금액에 제17조제3항 각 호 외의 부분 단서가 적용되는 배당소득금액이 합산되어 있는 경우에는 같은 항 각 호 외의 부분 단

In [12]:
# retrueved_docs[0].page_content
retrueved_doc = "\n\n--\n\n".join([doc.page_content for doc in retrueved_docs])

In [13]:
# query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
# retrueved_docs = database.similarity_search(query, k=3) 와 아래코드는 동일함

retruever = database.as_retriever(
    search_kwargs={"k":3}
)
retrueved_docs = retruever.invoke(query)

In [14]:
retrueved_docs[0].page_content

'1. 총급여액이 3천 300만원 이하인 경우: 74만원\n\n2. 총급여액이 3천 300만원 초과 7천만원 이하인 경우: 74만원 - [(총급여액 - 3천 300만원) × 8/1000]. 다만, 위 금액이 66만원보다 적은 경우에는 66만원으로 한다.\n\n3. 총급여액이 7천만원 초과 1억2천만원 이하인 경우: 66만원 - [(총급여액 - 7천만원) × 1/2]. 다만, 위 금액이 50만원보다 적은 경우에는 50만원으로 한다.\n\n4. 총급여액이 1억2천만원을 초과하는 경우: 50만원 - [(총급여액 - 1억2천만원) × 1/2]. 다만, 위 금액이 20만원보다 적은 경우에는 20만원으로 한다.\n\n③ 일용근로자의 근로소득에 대해서 제134조제3항에 따른 원천징수를 하는 경우에는 해당 근로소득에 대한 산출세액의 100분의 55에 해당하는 금액을 그 산출세액에서 공제한다.<개정 2014. 1. 1.>\n\n[전문개정 2012. 1. 1.]\n\n\n\n제59조의2(자녀세액공제) ①종합소득이 있는 거주자의 기본공제대상자에 해당하는 자녀(입양자 및 위탁아동을 포함하며, 이하 이 조에서 “공제대상자녀”라 한다) 및 손자녀로서 8세 이상의 사람에 대해서는 다음 각 호의 구분에 따른 금액을 종합소득산출세액에서 공제한다. <개정 2015. 5. 13., 2017. 12. 19., 2018. 12. 31., 2019. 12. 31., 2022. 12. 31., 2023. 12. 31., 2024. 12. 31.>\n\n1. 1명인 경우: 연 25만원\n\n2. 2명인 경우: 연 55만원\n\n3. 3명 이상인 경우: 연 55만원과 2명을 초과하는 1명당 연 40만원을 합한 금액\n\n② 삭제<2017. 12. 19.>\n\n③ 해당 과세기간에 출산하거나 입양 신고한 공제대상자녀가 있는 경우 다음 각 호의 구분에 따른 금액을 종합소득산출세액에서 공제한다.<신설 2015. 5. 13., 2016. 12. 20.>\n\n1. 출산하거나 입양 신고한 공제대상자녀가 첫째인 경

# 3. 답변 생성

In [15]:
# gpt 4.1 mini
from openai import OpenAI
client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role":"system", "content":"당신은 최고의 한국 소득세법 전문가입니다."},
        {
            "role":"user",
            "content":f"""- [content]를 참고해서 사용자의 질문에 10줄 이내로 답변해 주세요
            - [content]:{retrueved_doc}
            - 질문:{query}"""
        }
    ],
    temperature=0.2
)

In [16]:
print(response.choices[0].message.content)

연봉 5천만원인 직장인의 소득세 계산은 다음과 같습니다.

1. 종합소득 과세표준 구간: 1,400만원 초과 5,000만원 이하  
2. 세율 적용: 84만원 + (5,000만원 - 1,400만원) × 15% = 84만원 + 540만원 = 624만원  
3. 근로소득세액공제:  
   - 총급여 5,000만원은 3,300만원 초과 7,000만원 이하 구간  
   - 공제액 = 74만원 - (5,000만원 - 3,300만원) × 8/1000 = 74만원 - 13.6만원 = 60.4만원  
   - 다만, 최소 66만원 적용 → 공제액은 66만원  
4. 최종 소득세 = 624만원 - 66만원 = 558만원

따라서, 연봉 5천만원 직장인의 소득세는 약 558만원입니다.


In [26]:
# solar pro2
from openai import OpenAI
import os
client = OpenAI(
    api_key=os.getenv("UPSTAGE_API_KEY"),
    base_url="https://api.upstage.ai/v1"
)
response = client.chat.completions.create(
    model="solar-pro2",
    messages=[
        {"role":"system", "content":"당신은 최고의 한국 소득세법 전문가입니다."},
        {
            "role":"user",
            "content":f"""- [content]를 참고해서 사용자의 질문에 10줄 이내로 답변해 주세요
            - [content]:{retrueved_doc}
            - 질문:{query}"""
        }
    ],
    temperature=0.2
)

In [27]:
print(response.choices[0].message.content)

연봉 5천만원의 종합소득과세표준이 5,000만원일 경우, 소득세는 다음과 같이 계산됩니다:  

1. **5,000만원 초과 구간 적용**:  
   - 5,000만원 이하: 624만원 (5,000만원 × 24% - 624만원)  
   - 초과 금액(0원): 없음  
   - **산출세액 = 624만원**  

2. **공제 적용 후 최종 세액**:  
   - 근로소득공제, 자녀세액공제 등 추가 공제 가능 여부에 따라 변동될 수 있습니다.  

※ 정확한 세액은 공제 항목 및 과세표준에 따라 달라질 수 있으므로, 정확한 계산을 위해서는 세무사와 상담이 필요합니다.  

(답변: 10줄 이내 요약)  

**답변**:  
5,000만원 과세표준 시 기본 산출세액은 624만원입니다. 단, 근로소득공제, 자녀세액공제 등 적용 시 실제 납부세액은 감소할 수 있습니다. 정확한 금액은 공제 내역을 확인해야 합니다.


# 4. langchain 전달

In [19]:
from langchain_upstage import ChatUpstage, UpstageEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_pinecone import PineconeVectorStore
from dotenv import load_dotenv

# 1. LLM과 임베딩 초기화
load_dotenv()
# llm = ChatOpenAI(model = "gpt-4.1-mini")
from langchain_upstage import ChatUpstage
llm = ChatUpstage(model="solar-pro2")

embedding = UpstageEmbeddings(model="solar-embedding-1-large")

# 2. # 업로드한 벡터db를 가져올 때
vectorstore = PineconeVectorStore(
    embedding=embedding,# 질문을 임베딩하여 유사도 검색
    index_name=index_name
)
# 3. Retriever 생성
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":4})
# 4. 프롬프트 템플릿
template = f"""당신은 최고의 한국 소득세 전문가입니다.
다음 문맥을 참고하여 질문에 답하세요.
답을 모르면 모른다고 답하세요.
최대 3문장으로 간결하게 답변하세요.
질문 : {{query}}
문맥 : {{context}}
답변 : """
prompt = ChatPromptTemplate.from_template(template)
# 5. 검색된 document를 텍스트로 변환하는 함수
def format_documents(documents):
    return  "\n\n---\n\n".join([doc.page_content for doc in documents])

In [18]:
# 6. RAG 체인 구성(LCEL 방식)
from langchain_core.runnables import RunnablePassthrough # {"query":"~"}=>"~"
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문 그대로 전달
    }
    | prompt # prompt에 cdontext와 query 변수 주입
    | llm 
    | StrOutputParser()
)
# 7. 실행
query ="연봉 5천만원인 직장인의 소득세는 얼마인가요?"
rag_chain.invoke(query)

'연봉 5천만원 직장인의 근로소득세액공제는 66만원입니다(제59조 제2항 2호). 다만, 이는 근로소득세액공제액이며, 실제 소득세는 종합소득산출세액(제55조) 계산 후 자녀세액공제(제59조의2) 등을 적용해야 하므로 추가 정보 없이 정확한 세액을 산출할 수 없습니다.  \n\n(답변 요약: 근로소득세액공제는 66만원, 종합소득산출세액은 별도 계산 필요)'

# 5. 키워드 사전 활용

In [20]:
import numpy as np
def cosine_similarity(vec1, vec2):
    """두 백터 사이의 코사인 유사도 계산"""
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1) 
    norm_vec2 = np.linalg.norm(vec2)
    if norm_vec1==0 or norm_vec2==0:
        return 0.0
    return dot_product / (norm_vec1*norm_vec2)
embedding = UpstageEmbeddings(model="solar-embedding-1-large")

In [25]:
vec1 = embedding.embed_query("총급여")
vec2 = embedding.embed_query("연봉")
print("총급여와 연봉의 유사도 :", cosine_similarity(vec1, vec2))

총급여와 연봉의 유사도 : 0.6872200519738804


In [28]:
vec1 = embedding.embed_query("종합소득")
vec2 = embedding.embed_query("연봉")
print("종합소득과500 연봉의 유사도 :", cosine_similarity(vec1, vec2))

종합소득와 연봉의 유사도 : 0.56632913237249


In [29]:
vec1 = embedding.embed_query("직장인")
vec2 = embedding.embed_query("거주자")
print("직장인과 거주자의 유사도 :", cosine_similarity(vec1, vec2))

직작인와 거주자의 유사도 : 0.5400838658865784


In [30]:
vec1 = embedding.embed_query("5천만원")
vec2 = embedding.embed_query("5,000천만원")
print("5천만원과 5,000만원의 유사도 :", cosine_similarity(vec1, vec2))

5천만원과 5,000만원의 유사도 : 0.932639365145343


In [31]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrueved_docs = vectorstore.similarity_search(query, k=2)
retrueved_docs

[Document(id='0c1e7b60-37c8-48a0-9274-f2e72fa6373b', metadata={'source': 'data/소득세법_with_markdown.docx'}, page_content='1. 총급여액이 3천 300만원 이하인 경우: 74만원\n\n2. 총급여액이 3천 300만원 초과 7천만원 이하인 경우: 74만원 - [(총급여액 - 3천 300만원) × 8/1000]. 다만, 위 금액이 66만원보다 적은 경우에는 66만원으로 한다.\n\n3. 총급여액이 7천만원 초과 1억2천만원 이하인 경우: 66만원 - [(총급여액 - 7천만원) × 1/2]. 다만, 위 금액이 50만원보다 적은 경우에는 50만원으로 한다.\n\n4. 총급여액이 1억2천만원을 초과하는 경우: 50만원 - [(총급여액 - 1억2천만원) × 1/2]. 다만, 위 금액이 20만원보다 적은 경우에는 20만원으로 한다.\n\n③ 일용근로자의 근로소득에 대해서 제134조제3항에 따른 원천징수를 하는 경우에는 해당 근로소득에 대한 산출세액의 100분의 55에 해당하는 금액을 그 산출세액에서 공제한다.<개정 2014. 1. 1.>\n\n[전문개정 2012. 1. 1.]\n\n\n\n제59조의2(자녀세액공제) ①종합소득이 있는 거주자의 기본공제대상자에 해당하는 자녀(입양자 및 위탁아동을 포함하며, 이하 이 조에서 “공제대상자녀”라 한다) 및 손자녀로서 8세 이상의 사람에 대해서는 다음 각 호의 구분에 따른 금액을 종합소득산출세액에서 공제한다. <개정 2015. 5. 13., 2017. 12. 19., 2018. 12. 31., 2019. 12. 31., 2022. 12. 31., 2023. 12. 31., 2024. 12. 31.>\n\n1. 1명인 경우: 연 25만원\n\n2. 2명인 경우: 연 55만원\n\n3. 3명 이상인 경우: 연 55만원과 2명을 초과하는 1명당 연 40만원을 합한 금액\n\n② 삭제<2017. 12. 19.>\n\n③ 해당 과세기간에 출산하거나 입양

In [32]:
query = "종합소득 5000만원인 거주자의 소득세는 얼마인가요?"
retrueved_docs = vectorstore.similarity_search(query, k=2)
retrueved_docs

[Document(id='48b7893c-e71f-4158-95cc-cd85cd5a98bc', metadata={'source': 'data/소득세법_with_markdown.docx'}, page_content='제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n| 종합소득 과세표준 | 세율 |\n\n|---|---|\n\n| 1,400만원 이하 | 과세표준의 6퍼센트 |\n\n| 1,400만원 초과 5,000만원 이하 | 84만원 + (1,400만원을 초과하는 금액의 15퍼센트) |\n\n| 5,000만원 초과 8,800만원 이하 | 624만원 + (5,000만원을 초과하는 금액의 24퍼센트) |\n\n| 8,800만원 초과 1억5천만원 이하 | 1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트) |\n\n| 1억5천만원 초과 3억원 이하 | 3,706만원 + (1억5천만원을 초과하는 금액의 38퍼센트) |\n\n| 3억원 초과 5억원 이하 | 9,406만원 + (3억원을 초과하는 금액의 40퍼센트) |\n\n| 5억원 초과 10억원 이하 | 1억7,406만원 + (5억원을 초과하는 금액의 42퍼센트) |\n\n| 10억원 초과 | 3억8,406만원 + (10억원을 초과하는 금액의 45퍼센트) |\n\n\n\n\n\n② 거주자의 퇴직소득에 대한 소득세는 다음 각 호의 순서에 따라 계산한 금액(이하 “퇴직소득 산출세액”이라 한다)으로 한다.<개정 2013. 1. 1., 2014. 12. 23.>\n\n1. 해당 과세기간의 퇴직소득과세표준에 제1항의 세율을 적용하여 계산한 금액\n\n2. 제1호의 금액을 12로 나눈 금액에 근속연수를 곱한 금액\n\n3. 삭제<2014. 12.

In [54]:
# 사람을 나타내는 표현 -> 거주자로 변경 / 5천만원 -> 5,000만원 / 연봉 -> 총급여
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini")
dictionary = ["직장인 -> 거주자", "5천만원->5,000만원", "연봉->총급여"]
prompt = ChatPromptTemplate.from_template(f"""사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해 주세요. 
만약 변경할 필요가 없을 경우, 사용자의 질문을 변경하지 않아도 됩니다.
질문만 리턴해 주세요.
사전: {dictionary}
질문: {{question}}""")

In [55]:
parser = StrOutputParser()
parser.invoke(llm.invoke(prompt.invoke({"question":"소득이 높은 남자가 있습니다."})))

'소득이 높은 남자가 있습니다.'

In [56]:
dictionary_chain = prompt | llm | StrOutputParser()
dictionary_chain.invoke({"question":"소득이 높은 남자가 5천만원을 가지고 있습니다."})

'소득이 높은 남자가 5,000만원을 가지고 있습니다.'

In [57]:
dictionary_chain.invoke("연봉 5천만원인 직장인의 소득세는 얼마예요?")

'총급여 5,000만원인 거주자의 소득세는 얼마예요?'

In [58]:
# rag_chain.invoke("연봉 5천만원인 직장인의 소득세는 얼마예요?")
final_chain = dictionary_chain | rag_chain

In [59]:
final_chain.invoke("연봉 5천만원인 직장인의 소득세는 얼마예요?")

'총급여 5,000만원의 경우 두 번째 구간(3,300만원 초과 7,000만원 이하)에 해당하며, 소득세는 다음과 같이 계산됩니다:  \n**74만원 - (5,000만원 - 3,300만원) × 8/1000 = 74만원 - 1,440만원 × 0.008 = 74만원 - 11.52만원 = 62.48만원**  \n단, 최소 금액 조건(66만원 미만 시 66만원)에 따라 **66만원**이 적용됩니다.  \n\n(추가 공제 항목(자녀세액공제 등)은 별도 적용 필요)'

In [ ]:
# 입력 {"qustion":"질문"} -> dictionay_chain이 질문 멘트를 개선
# -> 개선된 질문을 rag_chain에 전달 -> retiever가 관련 문서 검색 -> format_documents
# -> 완성된 prompt를 llm에 전달하여 답변 생성 -> 문자만 추출해서 최종 답변